# Fine-tune Qwen-Image-Edit for SMT Sim2Real

Turn simulated **surface-mount technology (SMT)** component crops into images with your chosen **automated optical inspection (AOI)** appearance. You provide aligned source/target pairs; this notebook trains a **LoRA** adapter and compares its output against the unmodified base model.

**Start with `MODE = "smoke"`.** This checks the whole path with two training pairs and ten optimizer steps. Then set `MODE = "full"` and choose a fresh experiment name. A smoke test is not a trained model or a quality benchmark.

**Minimum GPU:** one NVIDIA GPU with **80 GB or more VRAM and native BF16 support** on Linux x86_64. A100 80 GB and H100 80 GB are compatible targets but have not been physically tested with this tutorial.

**Tested setup:** an RTX PRO 6000 Blackwell (96 GB), Ubuntu 24.04.4, Python 3.12.3, driver 590.48.01, and 64 GB host RAM with 30 GiB configured swap, using a fresh virtual environment. The complete smoke workflow includes fresh inference, fusion and offline reload. Successful execution does not certify inspection quality. Provide your own aligned source/target images.

You will produce: a checked data manifest → cached features → LoRA checkpoints → held-out comparisons → fresh inference → an optional fused model. All outputs go to the run directory you choose.

[Setup, terminal commands, Slurm and troubleshooting](README.md)

## Table of contents

- [Overview](#overview)
- [1. What is being trained?](#model)
- [2. Environment](#environment)
- [3. Choose paths and a run mode](#configure)
- [4. Prepare aligned pairs and split by source group](#data)
- [5. Fix the editing instruction](#prompt)
- [6. Check the GPU and download a pinned base model](#preflight)
- [7. Cache image latents and prompt embeddings](#cache)
- [8. Train the LoRA](#train)
- [9. Compare against the unmodified base on held-out data](#evaluate)
- [10. Fresh inference on your own source crop](#inference)
- [11. Optional: export a standalone fused pipeline](#export)
- [12. Cluster execution and troubleshooting](#cluster)


<a id="overview"></a>

## Overview

**Goal:** adapt the original Qwen-Image-Edit model so simulated SMT component crops resemble your chosen AOI lighting and surface appearance. The model should preserve component geometry, markings, and defects; the evaluation steps help you check whether it does.

**You provide:** aligned source/target RGB image pairs, a CSV identifying source groups and train/validation/test splits, and an instruction describing the target appearance. Targets can be registered real photographs or reviewed teacher-model outputs. No training dataset is downloaded by this notebook.

**You need:** a Linux x86_64 host with one NVIDIA GPU with at least 80 GB VRAM and native BF16, the isolated Python 3.12 environment described below, at least 200 GB free disk, and these four tutorial files together. The [README minimum requirements](README.md#minimum-requirements) cover host RAM, drivers and GPU compatibility; no other `pcb-aoi` code is required.

| Stage | What happens | What you get |
|---|---|---|
| Prepare | Validate pairs, separate source groups, inspect alignment, and fix the prompt. | A checked manifest and repeatable training inputs. |
| Cache | Frozen encoders turn training images and instructions into reusable features. | Train-only cached features with data/model provenance. |
| Train | Learn LoRA updates while the large base model stays frozen. | Adapter checkpoints, loss logs, timing, and GPU-memory measurements. |
| Evaluate | Generate base and adapted predictions using identical inputs and seeds. | Held-out comparisons for reviewing appearance, geometry, markings, and defects. |
| Use and export | Run a new source crop; optionally fuse the adapter and reload the export offline. | A prediction and, if requested, a standalone full model. |

**Start with smoke mode.** It exercises the training and inference path quickly. Change to full mode only after checking your data and smoke outputs; use a fresh experiment name.

| Mode | Training examples cached | Optimizer steps | Purpose |
|---|---|---:|---|
| `smoke` — default | First 2 training pairs, without augmentation. | 10 | Check setup, training, checkpoint loading, and inference. It does not establish output quality. |
| `full` | All training pairs, with 7 paired variants per pair. | 3,000 | Starting fine-tuning recipe; select and assess the resulting adapter on held-out data. |

Fresh-image inference and full-model export have explicit switches later in the notebook. For routine regression checks, reuse smoke mode in a new run directory; you do not need to repeat 3,000 steps. All generated files go under the paths you choose in section 3.


<a id="model"></a>

## 1. What is being trained?

**LoRA** means *low-rank adaptation*: learn a relatively small set of extra weights while leaving the large pretrained model unchanged. Save those extra weights as an adapter and load them alongside the same base model later.

- **Input:** a rendered/source RGB crop plus an editing instruction.
- **Target:** the same view and geometry with the desired inspection appearance. A target can be a registered real photograph or a reviewed teacher-model output; mark which it is.
- **VAE:** compresses input and target images into latent tensors—compact numerical representations of the images. Frozen.
- **Qwen2.5-VL:** reads the input image and instruction together. Its embeddings are image-specific, even when every example uses the same prompt. Frozen.
- **Diffusion transformer:** learns how to generate the target while conditioned on the source. Its original weights stay frozen; LoRA adds trainable low-rank updates to attention and feed-forward projections.
- **Scheduler:** controls the denoising trajectory during inference. Training samples a noise level with the corresponding resolution-aware shift.

Training adds noise to the target latent, gives the model that noisy target alongside the clean input latent, and asks it to predict the noise-minus-target direction. Only target tokens contribute to the flow-matching loss.

This notebook uses the original **`Qwen/Qwen-Image-Edit` / `QwenImageEditPipeline`**. 2509/2511 Plus models have different conditioning contracts. Training and evaluation operate on SMT component crops. Geometry preservation is a goal to verify, not a guaranteed property.

<a id="environment"></a>

## 2. Environment

Use a **Linux x86_64 GPU host** meeting these requirements:

| Requirement | Configuration |
|---|---|
| GPU | One full NVIDIA GPU with **80 GB or more VRAM and native BF16**. A100 80 GB, H100 80 GB, and RTX PRO 6000 Blackwell 96 GB are the target configurations. A100 40 GB and smaller MIG slices are excluded; this implementation does not pool memory across cards. |
| Host RAM | 64 GB minimum, 128 GB recommended. The tested 64 GB host had 30 GiB swap configured; 64 GB without swap has not been verified. |
| Disk | At least 200 GB free for packages, base weights, cache and optional export. |
| Software | Python 3.12 with `venv` and `pip`, Git, and NVIDIA driver 570.124.06 or newer with support for the specific GPU. Initial downloads need internet access. |

Only RTX PRO 6000 Blackwell has been physically tested with this tutorial. The pinned PyTorch build supports the A100/H100 architectures. Other Blackwell GPUs require 80 GB or more VRAM, Linux x86_64, compatible driver/wheel support, and a passing preflight plus smoke run; support is not assumed for every Blackwell or ARM system. See the [GPU matrix and source references](README.md#minimum-requirements).

On the **GPU machine**, open a terminal in this tutorial folder and create a separate Python 3.12 environment. Inside `pcb-aoi`, first run `cd docs/tutorials/sim2real-qwen-image-edit`. A standalone copy of the four tutorial files also works.

```bash
python3.12 -m venv .venv-smt
source .venv-smt/bin/activate
python -m pip install --upgrade pip
python -m pip install torch==2.10.0 torchvision==0.25.0 --index-url https://download.pytorch.org/whl/cu128
python -m pip install -r requirements.txt
python -m pip check
nvidia-smi --query-gpu=index,uuid,name,memory.total --format=csv
export CUDA_VISIBLE_DEVICES=GPU-REPLACE_WITH_UUID  # Copy the chosen GPU UUID above.
python -m jupyter lab fine_tune_smt.ipynb
```

Choose this environment as the notebook kernel. Do not install these dependencies into Isaac Sim's Python. The CUDA 12.8 wheel includes the target Ampere, Hopper and Blackwell architectures. Native PyTorch scaled dot-product attention (SDPA) selects an available GPU backend; no separate FlashAttention compilation is needed. See the [PyTorch installation reference](https://pytorch.org/get-started/previous-versions/#v2100).

The tested host has 64 GB RAM, 30 GiB configured swap and a 96 GB GPU. Training allocated approximately 41 GiB of GPU memory; inference used approximately 58 GiB. These are observations; the supported recipe still requires an 80 GB or larger GPU, with headroom for crop-dependent memory use. Allow ~200 GB free disk, including ~54 GiB for the base and another ~54 GiB for optional fusion. Caching omits the transformer; training omits the text encoder/VAE. Inference loads the full pipeline and may use more GPU memory. Each stage runs as a separate process so its memory is released.

On a machine with multiple GPUs, select the training GPU by copying its UUID from `nvidia-smi`. CUDA and `nvidia-smi` can enumerate cards in different orders; using the UUID avoids accidentally selecting a display card. Keep the scheduler-provided `CUDA_VISIBLE_DEVICES` when running on Slurm. The preflight checks native BF16 and the selected GPU's memory, reports the wheel's compiled architectures, and exercises BF16 attention in both directions. It prints the selected GPU and free memory; the subsequent smoke run verifies the full workload.


In [ ]:
import csv
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

# Run Jupyter from this directory or from the repository root.
TUTORIAL = Path.cwd()
if not (TUTORIAL / "smt_lora.py").is_file():
    TUTORIAL = TUTORIAL / "docs/tutorials/sim2real-qwen-image-edit"
TUTORIAL = TUTORIAL.resolve()
HELPER = TUTORIAL / "smt_lora.py"
assert HELPER.is_file(), "Start Jupyter from the repository root or tutorial directory."
assert sys.version_info >= (3, 12), (
    "Use Python 3.12 or newer; setup targets Python 3.12."
)

spec = importlib.util.spec_from_file_location("smt_lora", HELPER)
smt = importlib.util.module_from_spec(spec)
spec.loader.exec_module(smt)


def run(*arguments: object) -> None:
    """Run one stage in the notebook's Python environment and stop on failure."""
    subprocess.run(
        [sys.executable, "-u", str(HELPER), *map(str, arguments)], check=True
    )


print("Python:", sys.executable)
print("Tutorial:", TUTORIAL)

<a id="configure"></a>

## 3. Choose paths and a run mode

Use absolute paths on the GPU host. Keep your data and results outside the source checkout. `smoke` and `full` get different directories so a two-pair cache cannot accidentally become a full training cache.

A completed training directory cannot be overwritten. If you change the dataset, prompt, model revision, or code, choose a new experiment name. This tutorial exports adapter checkpoints; it does not resume optimizer state.

In [ ]:
DATA = Path("/data/my_smt_data")  # Change: your paired images and pairs.csv.
RESULTS = Path("/data/smt_runs")  # Change: writable output storage.
MODEL_CACHE = Path("/data/hf_cache")  # Change: enough room for the base model.
EXPERIMENT = "experiment-001"
MODE = "smoke"  # "smoke" or "full".
assert MODE in {"smoke", "full"}
assert DATA.is_absolute() and RESULTS.is_absolute() and MODEL_CACHE.is_absolute()
assert DATA.is_dir(), f"Create your paired dataset first: {DATA}"
RUN = RESULTS / f"{EXPERIMENT}-{MODE}"
RUN.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE)

MAX_STEPS = 10 if MODE == "smoke" else 3000
SAVE_EVERY = 10 if MODE == "smoke" else 500
AUGMENT = "none" if MODE == "smoke" else "rotshift"
print("Run directory:", RUN)
print("Steps:", MAX_STEPS, "Augmentation:", AUGMENT)

<a id="data"></a>

## 4. Prepare aligned pairs and split by source group

Prepare your own data:

```text
my_smt_data/
  input/component_a.png
  target/component_a.png
  input/component_b.png
  target/component_b.png
  ...
  pairs.csv
```

The corresponding images must show the same component, camera view, crop and geometry, with matching pixel dimensions. Normalize EXIF orientation and register real photographs to the renders before this step. The validator can check dimensions; **you must inspect alignment**.

Source crops can come from CDF rendering/cropping. That supplies only the input side. You still need your own registered photographs or approved, reviewed pseudo-targets. Using an unrelated photo or copying the input into the target folder does not teach the intended transfer.

Create `pairs.csv` with these columns and your actual filenames:

```csv
id,filename,group,split,target_kind
sample_a,component_a.png,source_board_a,train,real
sample_b,component_b.png,source_board_b,val,real
sample_c,component_c.png,source_board_c,test,real
```

The example is a format illustration, not a sufficient dataset. Use diverse crops across component types and appearance/defect conditions. `target_kind` is `real` or `pseudo`; pseudo-targets reflect a teacher model, not measured reality.

Keep related crops/augmentations/captures from the same board or render family in one **group**. Assign groups to train/val/test before augmentation; 60/20/20 is a starting point. Choose model settings on val and reserve test for a final assessment. Exact duplicates are checked, but near-duplicate or semantically related data still needs careful grouping.

In [ ]:
csv_path = DATA / "pairs.csv"
assert csv_path.is_file(), f"Create {csv_path} using the schema above."
manifest_rows = []
with csv_path.open(newline="", encoding="utf-8") as stream:
    reader = csv.DictReader(stream)
    required = {"id", "filename", "group", "split", "target_kind"}
    assert required.issubset(reader.fieldnames or []), (
        f"Missing CSV columns: {required}"
    )
    for row in reader:
        filename = row["filename"].strip()
        assert Path(filename).name == filename, (
            "Use flat input/target folders with plain filenames."
        )
        manifest_rows.append(
            {
                "id": row["id"].strip(),
                "input": f"input/{filename}",
                "target": f"target/{filename}",
                "group": row["group"].strip(),
                "split": row["split"].strip(),
                "target_kind": row["target_kind"].strip(),
            }
        )
MANIFEST = DATA / "pairs.jsonl"
contents = "".join(json.dumps(row) + "\n" for row in manifest_rows)
if MANIFEST.exists():
    assert MANIFEST.read_text() == contents, (
        "Manifest differs. Review changes before replacing it."
    )
else:
    MANIFEST.write_text(contents, encoding="utf-8")
run("validate", "--manifest", MANIFEST)
rows = smt._inventory(MANIFEST)
print("Checked", len(rows), "pairs")

### Inspect pairs before training

The left image is the source; the right image is its target. Check pad/lead alignment, component body boundaries, crop framing and markings. A size match is not proof of registration. Review more than the displayed examples, including every component family and unusual aspect ratio.

In [ ]:
from IPython.display import display
from PIL import Image, ImageDraw, ImageOps


def show_pair(row: dict) -> None:
    """Display an input/target pair with aspect ratio preserved."""
    panel = Image.new("RGB", (768, 410), "#20252b")
    draw = ImageDraw.Draw(panel)
    for column, key in enumerate(("input", "target")):
        with Image.open(row[key]) as image:
            preview = ImageOps.contain(image.convert("RGB"), (376, 376))
        panel.paste(preview, (column * 384 + (384 - preview.width) // 2, 28))
        draw.text((column * 384 + 8, 8), key, fill="white")
    print(row["id"], row["group"], row["split"], "target:", row["target_kind"])
    display(panel)


for split in ("train", "val", "test"):
    for row in [r for r in rows if r["split"] == split][:2]:
        show_pair(row)

<a id="prompt"></a>

## 5. Fix the editing instruction

Use one instruction that describes the style in your targets and explicitly asks to preserve source geometry. This example suits restrained orange/blue solder-light targets. Change it if your own targets use a different inspection appearance. The text is a training condition; it cannot enforce geometry by itself.

Use the same prompt when training and evaluating. Changing it requires rebuilding the cache because the image/text embeddings depend on both prompt and input image.

In [ ]:
PROMPT = (
    "Render this PCB component crop as an AOI photograph under raked solder lighting. "
    "Preserve the source geometry, component shape, crop, leads, pads, markings, and defects. "
    "Use orange-red and blue specular highlights on existing exposed metal and solder. "
    "Keep matte component bodies dark and non-emissive. "
    "Do not invent text, components, halos, glow, or decorative details."
)
PROMPT_FILE = RUN / "prompt.txt"
if PROMPT_FILE.exists():
    assert PROMPT_FILE.read_text() == PROMPT + "\n", (
        "Changed prompt: use a new experiment name."
    )
else:
    PROMPT_FILE.write_text(PROMPT + "\n", encoding="utf-8")
print(PROMPT)

<a id="preflight"></a>

## 6. Check the GPU and download a pinned base model

The preflight executes a tiny BF16 forward/backward operation and reports the GPU, CUDA runtime and installed versions. It does not guarantee that the full model fits; the smoke run tests that next.

Use the tested immutable base-model commit and keep that revision in the run folder. The helper loads that local snapshot without contacting the Hub; the download below is the explicit network step. For a pre-populated cache on an offline node, set `HF_HUB_OFFLINE=1` before starting Jupyter. Downloading the full model is substantial. Review the [base model's terms](https://huggingface.co/Qwen/Qwen-Image-Edit). If authentication is needed, log in with the Hugging Face CLI on the GPU host; do not paste a token into a notebook cell.

In [ ]:
run("preflight")
from huggingface_hub import snapshot_download
from huggingface_hub.constants import HF_HUB_OFFLINE

REVISION_FILE = RUN / "base-revision.txt"
if REVISION_FILE.exists():
    BASE_REVISION = REVISION_FILE.read_text().strip()
else:
    BASE_REVISION = smt.BASE_REVISION  # Tested immutable model revision.
    REVISION_FILE.write_text(BASE_REVISION + "\n", encoding="utf-8")
print("Pinned model:", smt.BASE_MODEL, BASE_REVISION)
MODEL_PATH = snapshot_download(
    smt.BASE_MODEL, revision=BASE_REVISION, local_files_only=HF_HUB_OFFLINE
)
print("Downloaded model:", MODEL_PATH)
(RUN / "environment.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True),
    encoding="utf-8",
)

<a id="cache"></a>

## 7. Cache image latents and prompt embeddings

Each source and target is resized to approximately **262,144 pixels (0.262 MP; 512×512 for square crops)**, with aspect ratio approximately preserved and dimensions snapped to multiples of 32. The VAE creates compact latents. Qwen2.5-VL creates a separate image-conditioned embedding for each example.

Full mode creates seven paired variants: original, rotations ±10°, and shifts ±5% in x/y. Both images get the same geometric operation; crops avoid padded borders. Disable augmentation if it invalidates orientation-sensitive content. Only **train** rows enter this cache.

The cache records image hashes, prompt, revision, code and environment. It is reused only when those match and all files are intact. An interrupted cache needs a new directory. The cache stage omits the diffusion transformer to reduce memory use.

In [ ]:
CACHE = RUN / "cache"
cache_command = [
    "cache",
    "--manifest",
    MANIFEST,
    "--prompt-file",
    PROMPT_FILE,
    "--revision",
    BASE_REVISION,
    "--output",
    CACHE,
    "--area",
    262144,
    "--augment",
    AUGMENT,
]
if MODE == "smoke":
    cache_command += ["--limit", 2]
run(*cache_command)
cache_info = json.loads((CACHE / "cache.json").read_text())
print("Cached samples:", len(cache_info["files"]))
print("Cache runtime:", cache_info.get("runtime"))

<a id="train"></a>

## 8. Train the LoRA

The default full recipe uses rank/alpha **16/16**, learning rate **1e-4**, **3,000 steps**, batch **1**, gradient checkpointing and BF16 frozen weights. Trainable LoRA parameters use FP32 for optimization. Checkpoints are saved every 500 steps and at the final step. These are starting values, not universal optima for your data.

Batch 1 supports variable crop aspect ratios. A step is one optimizer update, not an epoch. The script does not select a best checkpoint from loss alone, resume optimizer state, or distribute training across GPUs. A Slurm allocation runs the same single-GPU command.

In [ ]:
TRAIN = RUN / "train"
run(
    "train",
    "--cache",
    CACHE,
    "--output",
    TRAIN,
    "--steps",
    MAX_STEPS,
    "--save-every",
    SAVE_EVERY,
    "--rank",
    16,
    "--lr",
    1e-4,
    "--seed",
    0,
)
CHECKPOINT = TRAIN / f"lora-step-{MAX_STEPS:05d}"
assert (CHECKPOINT / "pytorch_lora_weights.safetensors").is_file()
loss_rows = [
    json.loads(line) for line in (TRAIN / "loss.jsonl").read_text().splitlines()
]
print("Checkpoint:", CHECKPOINT)
print("Last observed training record:", loss_rows[-1])

<a id="evaluate"></a>

## 9. Compare against the unmodified base on held-out data

The base and adapted model receive identical inputs, prompt, inference steps, CFG and random seed. Evaluate every important component family, not only visually appealing examples. Use `val` to select checkpoints, adapter strength, and training settings. Do not tune on `test`.

The native original Qwen pipeline defaults to approximately **1 MP inference**, unlike the 0.262 MP training cache. Passing output width/height alone does not change how the pipeline resizes the reference image. This tutorial keeps native inference behavior and records it. Qualify the resolution you will actually deploy; do not assume source pixel coordinates carry over unchanged.

In smoke mode the adapter has only ten updates. The expected success is that the comparison runs and writes images, not that it already improves realism.

In [ ]:
COMPARISON = RUN / f"val-step{MAX_STEPS:05d}"
compare_command = [
    "compare",
    "--manifest",
    MANIFEST,
    "--lora",
    CHECKPOINT,
    "--output",
    COMPARISON,
    "--split",
    "val",
    "--steps",
    30,
    "--cfg",
    4.0,
    "--seed",
    0,
    "--scale",
    1.0,
]
if MODE == "smoke":
    compare_command += ["--limit", 1]
run(*compare_command)

In [ ]:
comparison = json.loads((COMPARISON / "comparison.json").read_text())
for row in comparison["rows"][:6]:
    paths = [
        Path(row["input"]),
        Path(row["target"]),
        COMPARISON / f"{row['id']}__base.png",
        COMPARISON / f"{row['id']}__adapted.png",
    ]
    panel = Image.new("RGB", (1280, 350), "#20252b")
    draw = ImageDraw.Draw(panel)
    for column, (label, path) in enumerate(
        zip(("Input", "Target", "Base", "LoRA"), paths)
    ):
        with Image.open(path) as image:
            preview = ImageOps.contain(image.convert("RGB"), (312, 312))
        panel.paste(preview, (column * 320 + (320 - preview.width) // 2, 28))
        draw.text((column * 320 + 8, 8), label, fill="white")
    print(row["id"], "Target kind:", row["target_kind"])
    display(panel)

### Accept or reject the adapter

Review geometry, pad/lead counts, markings, body texture, highlight placement, and **defect preservation**. Reject repaired defects, invented pins, moved components and new text. Save review notes beside the predictions. Lower training loss or closer agreement with a pseudo-target is not proof of real-world accuracy.

For full training, compare earlier checkpoint directories and adapter scales (for example 0.5 and 1.0) using fresh output directories and **validation** data. Then run `compare --split test` once on untouched groups with your chosen checkpoint/settings. A downstream inspection-model assessment on real images is needed for claims about inspection performance.

A useful review record is `id,geometry_ok,defect_preserved,style_ok,notes`. Inspect the whole validation set; the gallery above shows only the first six examples.

<a id="inference"></a>

## 10. Fresh inference on your own source crop

Set a new image path, then enable the cell. This is a direct Diffusers example using your adapter and the exact base revision and prompt saved with it. It does not need a target image. Keep the inference settings aligned with what you validated.

In [ ]:
RUN_FRESH_INFERENCE = False
NEW_IMAGE = Path("/data/new_component.png")  # Change to an unseen source crop.

if RUN_FRESH_INFERENCE:
    import torch
    from diffusers import QwenImageEditPipeline

    assert NEW_IMAGE.is_file()
    info = json.loads((CHECKPOINT / "training.json").read_text())
    pipe = QwenImageEditPipeline.from_pretrained(
        smt._base_snapshot(info["revision"]),
        torch_dtype=torch.bfloat16,
        local_files_only=True,
    ).to("cuda")
    pipe.load_lora_weights(
        str(CHECKPOINT),
        weight_name="pytorch_lora_weights.safetensors",
        adapter_name="smt",
    )
    pipe.set_adapters("smt", adapter_weights=1.0)
    with Image.open(NEW_IMAGE) as image:
        image = image.convert("RGB")
    with torch.inference_mode():
        result = pipe(
            image=image,
            prompt=info["cache"]["signature"]["prompt"],
            negative_prompt=" ",
            true_cfg_scale=4.0,
            num_inference_steps=30,
            generator=torch.Generator(device="cuda").manual_seed(0),
        ).images[0]
    result.save(RUN / "fresh_prediction.png")
    print("Input size:", image.size, "Output size:", result.size)
    display(result)
    del pipe
    import gc

    gc.collect()
    torch.cuda.empty_cache()
else:
    print("Set NEW_IMAGE and RUN_FRESH_INFERENCE to generate an unseen crop.")

<a id="export"></a>

## 11. Optional: export a standalone fused pipeline

An adapter is small relative to the base model and needs that base at inference. **Fusion** merges its updates into a full pipeline that can be loaded on its own. It duplicates much of the base model on disk. Use the same adapter scale you validated.

Before deployment, reload the fused artifact and compare a fixed input/seed against the unfused adapter. BF16 fusion can introduce small numerical differences. This notebook's export is a local original-Qwen pipeline; serving it through a CDF endpoint is a separate integration task.

In [ ]:
EXPORT_FUSED = False
FUSED = RUN / "fused"
if EXPORT_FUSED:
    run("fuse", "--lora", CHECKPOINT, "--output", FUSED, "--scale", 1.0)
    assert (FUSED / "model_index.json").is_file()
    print("Exported:", FUSED)
else:
    print(
        "Adapter export is sufficient for normal use. Set EXPORT_FUSED to save a full model."
    )

### Reload the exported model offline

An export is only useful if it can be loaded again. This optional cell reloads the full pipeline from disk without the adapter or a network connection, then repeats the first validation prediction with the same input, prompt, seed, and inference settings. BF16 fusion rounds merged weights, so pixel-for-pixel equality is not guaranteed. Inspect both images; a small average pixel difference alone does not prove that markings or defects were preserved.

In [ ]:
if EXPORT_FUSED:
    import gc

    import numpy as np
    import torch
    from diffusers import QwenImageEditPipeline

    comparison = json.loads((COMPARISON / "comparison.json").read_text())
    row = comparison["rows"][0]
    pipe = QwenImageEditPipeline.from_pretrained(
        FUSED, torch_dtype=torch.bfloat16, local_files_only=True
    ).to("cuda")
    with Image.open(row["input"]) as image:
        image = image.convert("RGB")
    with torch.inference_mode():
        fused_result = pipe(
            image=image,
            prompt=comparison["prompt"],
            negative_prompt=" ",
            true_cfg_scale=comparison["cfg"],
            num_inference_steps=comparison["steps"],
            generator=torch.Generator(device="cuda").manual_seed(comparison["seed"]),
        ).images[0]
    fused_result.save(RUN / "fused_reload.png")
    with Image.open(COMPARISON / f"{row['id']}__adapted.png") as reference:
        reference = reference.convert("RGB")
        assert reference.size == fused_result.size
        difference = np.abs(
            np.asarray(reference, dtype=np.float32)
            - np.asarray(fused_result, dtype=np.float32)
        ).mean()
        print("Unfused adapter prediction:")
        display(reference)
    print("Reloaded fused prediction:")
    display(fused_result)
    print("Mean absolute RGB difference (0–255):", float(difference))
    del pipe
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("Export a fused model above to run the offline reload check.")

Keep the fixed prompt, chosen model revision, training settings, validation review and output-size behavior with the model. Copy only the artifact you intend to distribute; `training.json` and cache provenance include local paths and dataset metadata and should be reviewed before sharing.

<a id="cluster"></a>

## 12. Cluster execution and troubleshooting

The [README](README.md#run-the-same-steps-with-slurm) includes a sequential `sbatch` recipe using these same commands. Pre-download weights to a shared `HF_HOME`; compute nodes may have no internet. Request one suitable GPU, set your cluster account/partition, and monitor `squeue`, the job log and `sacct`.

If a run fails, fix its cause and choose fresh outputs. For OOM, release other GPU jobs, keep batch 1/checkpointing, and consider a lower cache area in a new run. Caching, training and inference have different memory needs. Never interpret a successful smoke test, falling loss, or an attractive generated example as a validated inspection dataset.

See [the full troubleshooting table](README.md#troubleshooting).